In [ ]:
!pip install accelerate bitsandbytes psutil

In [ ]:
from transformers import AutoProcessor, Gemma3nForConditionalGeneration, BitsAndBytesConfig
import torch
import librosa
import numpy as np

In [ ]:
###############################
# Memory cleaning

# import torch
# import gc

# torch.cuda.empty_cache()
# gc.collect() # python garbage collector
###############################

In [ ]:
model_id = "google/gemma-3n-e2b-it"

model = Gemma3nForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float32, # bfloat16 does not work, float16 does not work either, only float32
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True), # bitsAndBytes
    trust_remote_code=True
).eval()

model.gradient_checkpointing_enable()

In [ ]:
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
"""
RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.cuda.HalfTensor) should be the same
"""

# Fix:


In [ ]:
def process_audio_file(audio_path, prompt="Верни в ответ транкскрипцию услышанного"):
    audio, sr = librosa.load(audio_path, sr=16000)

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "Ты полезный ассистент, который может анализировать аудио."}]
        },
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": audio},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    generation = generation[0][input_len:]
    decoded = processor.decode(generation, skip_special_tokens=True)

    return decoded

In [ ]:
process_audio_file(audio_path="./chunk_2.wav")